# 📊 RAG Pipeline Evaluation — HUSC Admissions 2025

**Tiểu luận: So sánh hiệu năng 3 Embedding Models × 3 Route Strategies**

| Embedding | Route Strategies | Metrics |
|-----------|------------------|---------|
| BGE-M3 | PaddedRAG (vector-only) | Accuracy, Recall, Hallucination |
| Harrier 0.6B | GraphRAG (vector+graph) | Groundedness, Latency p50/p95 |
| Qwen3 0.6B | Auto-route (hybrid) | Per-category breakdown |

**Runtime:** Google Colab T4 GPU • **Restart kernel → Run All** to reproduce

## 1. Setup & Prerequisites

In [28]:
import os, sys, time
print(f'Python {sys.version}')
!nvidia-smi 2>&1 | head -5
import torch
assert torch.cuda.is_available(), 'GPU not enabled. Runtime > Change runtime type > T4 GPU'
print(f'GPU: {torch.cuda.get_device_name(0)}')

Python 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
Sun Apr 19 15:01:04 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
GPU: Tesla T4


In [29]:
# Load API key from Colab Secrets
from google.colab import userdata
os.environ['RAMCLOUDS_API_KEY'] = userdata.get('RAMCLOUDS_API_KEY')
_k = os.environ.get('RAMCLOUDS_API_KEY', '')
_masked = (_k[:4] + '...' + _k[-4:]) if len(_k) >= 8 else ('set-but-short' if _k else 'missing')
print(f'API key loaded: {bool(_k)} | key(masked): {_masked}')


API key loaded: True | key(masked): sk-d...yFEe


In [30]:
!git clone --depth 1 -b main https://github.com/TienDat11/husc-admission-chat-enrollment.git 2>&1 | tail -3
%cd husc-admission-chat-enrollment/rag2025
!pip install -r requirements.txt -q 2>&1 | tail -3
print('\n✅ Clone + install done')

fatal: destination path 'husc-admission-chat-enrollment' already exists and is not an empty directory.
/content/husc-admission-chat-enrollment/rag2025/husc-admission-chat-enrollment/rag2025

✅ Clone + install done


## 2. Configuration

**IMPORTANT:** Chọn embedding model bạn muốn test. Mỗi Colab chạy 1 embedding.

In [31]:
#@title Chọn Embedding Model {display-mode: "form"}
EMBEDDING_CHOICE = 'bge' #@param ['bge', 'harrier', 'qwen']

EMBEDDING_MAP = {
    'bge':     {'model': 'BAAI/bge-m3',                    'dim': 1024},
    'harrier': {'model': 'microsoft/harrier-oss-v1-0.6b',  'dim': 1024},
    'qwen':    {'model': 'Qwen/Qwen3-Embedding-0.6B',      'dim': 1024},
}

emb_cfg = EMBEDDING_MAP[EMBEDDING_CHOICE]

# Write .env
env_content = f"""RAMCLOUDS_API_KEY={os.environ['RAMCLOUDS_API_KEY']}
RAMCLOUDS_BASE_URL=https://ramclouds.me/v1
RAMCLOUDS_MODEL=Gemma 4 31B IT
EMBEDDING_PROVIDER={EMBEDDING_CHOICE}
EMBEDDING_MODEL={emb_cfg['model']}
EMBEDDING_DIM={emb_cfg['dim']}
USE_LANCEDB=true
LANCEDB_URI=./data/lancedb
LANCEDB_TABLE=rag2025_{EMBEDDING_CHOICE}
HF_HOME=/content/hf_cache
TRANSFORMERS_CACHE=/content/hf_cache/hub
RERANKER_ENABLED=false
GUARDRAIL_ENABLED=false
ERROR_EXPOSURE_MODE=dev
"""
with open('.env', 'w') as f:
    f.write(env_content)

# Also set in os.environ for immediate use
for line in env_content.strip().split('\n'):
    if '=' in line and not line.startswith('#'):
        k, v = line.split('=', 1)
        os.environ[k] = v

print(f'✅ Config: {EMBEDDING_CHOICE} ({emb_cfg["model"]}, dim={emb_cfg["dim"]})')


✅ Config: bge (BAAI/bge-m3, dim=1024)


## 3. Patch GPU + Verify LLM

In [32]:
# ═══════════════════════════════════════════════════════════════
# PATCH 1: GPU + Verbose API Logging
# ═══════════════════════════════════════════════════════════════
import fileinput, json, requests, time, sys, os

# --- Patch embedding.py for GPU ---
for line in fileinput.input('src/services/embedding.py', inplace=True):
    print(line.replace('device="cpu"', 'device="cuda"'), end='')
print('✅ Patched embedding.py → device=cuda')

# --- Ensure sys/time are at the TOP of the files ---
def ensure_imports(path):
    if not os.path.exists(path): return
    with open(path, 'r') as f: content = f.read()
    if 'import sys' not in content:
        with open(path, 'w') as f: f.write('import sys, time\n' + content)

ensure_imports('src/services/llm_client.py')
ensure_imports('src/services/ner_service.py')

# --- Patch llm_client.py with EXTREME logging ---
_llm_client_path = 'src/services/llm_client.py'
_llm_verbose_patch = '''
# === VERBOSE API LOGGING (injected by Colab) ===
import time as _time
import sys as _sys

try:
    _original_call_provider = UnifiedLLMClient._call_provider

    async def _verbose_call_provider(self, provider, messages, temperature, max_tokens, json_mode):
        _t0 = _time.perf_counter()
        _model = self._force_model or provider.model
        _msg_count = len(messages)
        _total_chars = sum(len(m.get("content", "")) for m in messages)
        print(f"\\n{'='*60}")
        print(f"▄ API CALL → [{provider.name}] model={_model}")
        print(f"   base_url={provider.base_url}")
        print(f"   messages={_msg_count} turns | total_chars={_total_chars}")
        print(f"   temperature={temperature} | max_tokens={max_tokens} | json_mode={json_mode}")
        _sys.stdout.flush()
        try:
            result = await _original_call_provider(self, provider, messages, temperature, max_tokens, json_mode)
            _elapsed = (_time.perf_counter() - _t0) * 1000
            _content_preview = (result.content or "")[:300]
            print(f"✅ API OK ← [{provider.name}/{result.model}] {_elapsed:.0f}ms")
            print(f"   response_len={len(result.content or '')} chars")
            print(f"   preview: {_content_preview[:150]}...")
            print(f"{'='*60}")
            _sys.stdout.flush()
            return result
        except Exception as exc:
            _elapsed = (_time.perf_counter() - _t0) * 1000
            print(f"❆ API FAIL ← [{provider.name}] {type(exc).__name__} {_elapsed:.0f}ms")
            print(f"   error_message={str(exc)[:300]}")
            print(f"{'='*60}")
            _sys.stdout.flush()
            raise

    UnifiedLLMClient._call_provider = _verbose_call_provider
except NameError:
    pass
'''
if os.path.exists(_llm_client_path):
    with open(_llm_client_path, 'r', encoding='utf-8') as f:
        content = f.read()
    if '_verbose_call_provider' not in content:
        with open(_llm_client_path, 'a', encoding='utf-8') as f:
            f.write(_llm_verbose_patch)
        print('✅ Patched llm_client.py with verbose logging')

# --- Patch ner_service.py with verbose logging ---
_ner_path = 'src/services/ner_service.py'
_ner_verbose_patch = '''
# === VERBOSE NER LOGGING (injected by Colab) ===
import time as _ner_time
import sys as _ner_sys

try:
    _orig_extract = NERService.extract

    async def _verbose_extract(self, chunk):
        _t0 = _ner_time.perf_counter()
        print(f"\\n▄ NER EXTRACT → chunk_id={chunk.chunk_id} | text_len={len(chunk.text)}")
        _ner_sys.stdout.flush()
        try:
            result = await _orig_extract(self, chunk)
            _elapsed = (_ner_time.perf_counter() - _t0) * 1000
            _n_ent = len(result.entities) if result.entities else 0
            _n_tri = len(result.triples) if result.triples else 0
            _err = getattr(result, 'error', None)
            if _err:
                print(f"☐ NER WARN ← chunk={chunk.chunk_id} {_elapsed:.0f}ms | error={_err}")
            else:
                print(f"✅ NER OK ← chunk={chunk.chunk_id} {_elapsed:.0f}ms | entities={_n_ent} | triples={_n_tri}")
            _ner_sys.stdout.flush()
            return result
        except Exception as exc:
            _elapsed = (_ner_time.perf_counter() - _t0) * 1000
            print(f"❆ NER FAIL ← chunk={chunk.chunk_id} {_elapsed:.0f}ms | {type(exc).__name__}: {str(exc)[:200]}")
            _ner_sys.stdout.flush()
            raise

    NERService.extract = _verbose_extract
except NameError:
    pass
'''
if os.path.exists(_ner_path):
    with open(_ner_path, 'r', encoding='utf-8') as f: content = f.read()
    if '_verbose_extract' not in content:
        with open(_ner_path, 'a', encoding='utf-8') as f:
            f.write(_ner_verbose_patch)
        print('✅ Patched ner_service.py with verbose logging')


✅ Patched embedding.py → device=cuda


## 4. Data Ingest → LanceDB + Knowledge Graph

In [33]:
import json
import subprocess
import os
import sys

# Check where the scripts actually are
if not os.path.exists('/content/husc-admission-chat-enrollment'):
    print("Repo not found. Re-cloning...")
    !git clone --depth 1 -b main https://github.com/TienDat11/husc-admission-chat-enrollment.git /content/husc-admission-chat-enrollment

# Find the correct base path
possible_paths = [
    '/content/husc-admission-chat-enrollment/rag2025',
    '/content/husc-admission-chat-enrollment',
    '/content/rag2025'
]

BASE_PATH = None
for p in possible_paths:
    if os.path.exists(os.path.join(p, 'scripts/ingest_lancedb.py')):
        BASE_PATH = p
        break

if BASE_PATH:
    %cd {BASE_PATH}
    print(f"Using BASE_PATH: {BASE_PATH}")
else:
    print("❌ Error: Could not find scripts directory!")

# Fix the SyntaxError: move __future__ to the very top and ensure sys is imported
ner_path = 'src/services/ner_service.py'
if os.path.exists(ner_path):
    with open(ner_path, 'r') as f:
        lines = f.readlines()

    # Remove any existing __future__ or 'import sys' we added wrongly
    new_lines = [l for l in lines if 'from __future__' not in l and 'import sys' != l.strip()]

    # Re-insert at the absolute beginning
    fixed_content = "from __future__ import annotations\nimport sys\n" + "".join(new_lines)

    with open(ner_path, 'w') as f:
        f.write(fixed_content)
    print("✅ Fixed SyntaxError and missing 'sys' import in ner_service.py")

# Data check
!ls -la data/chunked/ 2>/dev/null || echo "Data dir missing"
!echo "---"

def run_step(step_name, cmd):
    print(f'\n>>> {step_name}')
    proc = subprocess.run(cmd, capture_output=True, text=True, env=os.environ.copy())
    if proc.stdout: print(proc.stdout)
    if proc.stderr: print(proc.stderr)
    if proc.returncode != 0:
        print('❌ STEP_FAILED')
        raise RuntimeError(f'{step_name} failed')

# Execute steps
os.environ['HF_HOME'] = '/content/hf_cache'
run_step('Step 1: LanceDB Ingest', ['python', 'scripts/ingest_lancedb.py'])
run_step('Step 2: Build Knowledge Graph', ['python', 'scripts/build_graph.py'])
run_step('Step 3: Verification', ['python', 'scripts/preflight_check.py'])

/content/husc-admission-chat-enrollment/rag2025
Using BASE_PATH: /content/husc-admission-chat-enrollment/rag2025
✅ Fixed SyntaxError and missing 'sys' import in ner_service.py
total 372
drwxr-xr-x 2 root root  4096 Apr 19 14:14 .
drwxr-xr-x 8 root root  4096 Apr 19 14:16 ..
-rw-r--r-- 1 root root 60054 Apr 19 14:14 chunked_10_enhanced.jsonl
-rw-r--r-- 1 root root 52830 Apr 19 14:14 chunked_10.jsonl
-rw-r--r-- 1 root root 20083 Apr 19 14:14 chunked_11.jsonl
-rw-r--r-- 1 root root 43984 Apr 19 14:14 chunked_1.jsonl
-rw-r--r-- 1 root root  3933 Apr 19 14:14 chunked_2.jsonl
-rw-r--r-- 1 root root 87110 Apr 19 14:14 chunked_3.jsonl
-rw-r--r-- 1 root root  4980 Apr 19 14:14 chunked_4.jsonl
-rw-r--r-- 1 root root 21710 Apr 19 14:14 chunked_5.jsonl
-rw-r--r-- 1 root root  6070 Apr 19 14:14 chunked_6.jsonl
-rw-r--r-- 1 root root     0 Apr 19 14:14 chunked_7.jsonl
-rw-r--r-- 1 root root 34526 Apr 19 14:14 chunked_8.jsonl
-rw-r--r-- 1 root root  8400 Apr 19 14:14 chunked_9.jsonl
-rw-r--r-- 1 root

RuntimeError: Step 2: Build Knowledge Graph failed

## 5. Start Pipeline Server

In [ ]:
import subprocess, time

proc = subprocess.Popen(
    ['python', '-m', 'uvicorn', 'src.main:app', '--host', '0.0.0.0', '--port', '8000'],
    stdout=open('/content/server.log', 'w'),
    stderr=subprocess.STDOUT,
)
time.sleep(20)

try:
    r = requests.get('http://127.0.0.1:8000/health', timeout=10)
    print(f'✅ Server running: {r.status_code}')
except Exception as e:
    print(f'❌ Server failed: {e}')
    !tail -40 /content/server.log

## 6. Full Evaluation — 3 Routes × 50 Questions

Tests: `padded_rag` (vector-only), `graph_rag` (vector+graph), `auto` (hybrid router)

In [ ]:
import json, time, requests, os, traceback
sys.path.insert(0, '/content/husc-admission-chat-enrollment/packages/rag-chatbot-husc/src')

from notebooks.eval_core import (
    load_test_questions, normalize_pipeline_output,
    exact_correctness, retrieval_recall_proxy, hallucination_flag,
    GROUNDING_THRESHOLD,
)

EMB = os.environ['EMBEDDING_PROVIDER']
BASE_URL = 'http://127.0.0.1:8000'
ROUTES = ['padded_rag', 'graph_rag', None]  # None = auto-route
ROUTE_NAMES = {None: 'auto', 'padded_rag': 'padded_rag', 'graph_rag': 'graph_rag'}

rows, used_path = load_test_questions(
    'results/test_questions.json',
    'backup_mail_package_2026/python_project/rag2025/results/test_questions.json',
)
print(f'Loaded {len(rows)} questions from {used_path}')

os.makedirs('results/colab_eval', exist_ok=True)

all_results = []

for route in ROUTES:
    route_name = ROUTE_NAMES[route]
    print(f'\n{"="*60}')
    print(f'>>> Route: {route_name} | Embedding: {EMB}')
    print(f'{"="*60}')

    for i, item in enumerate(rows):
        q = item['question']
        start = time.perf_counter()

        # ─── VERBOSE QUERY LOGGING ───
        print(f'\n  [{i+1}/{len(rows)}] Q: {q[:80]}...')
        sys.stdout.flush()

        try:
            payload = {'query': q, 'top_k': 5}
            if route is not None:
                payload['force_route'] = route

            resp = requests.post(f'{BASE_URL}/v2/query', json=payload, timeout=120)
            elapsed = time.perf_counter() - start

            # ─── VERBOSE RESPONSE LOGGING ───
            print(f'    HTTP {resp.status_code} | {elapsed*1000:.0f}ms')

            if resp.status_code != 200:
                print(f'    ❌ ERROR BODY: {resp.text[:500]}')
                raise RuntimeError(f'HTTP {resp.status_code}: {resp.text[:200]}')

            raw = resp.json()

            # Log route actually used + chunks retrieved
            _actual_route = raw.get('route', raw.get('metadata', {}).get('route', '?'))
            _chunks = raw.get('chunks', raw.get('context', []))
            _n_chunks = len(_chunks) if isinstance(_chunks, list) else 0
            _graph_nodes = raw.get('metadata', {}).get('graph_nodes', raw.get('graph_nodes', []))
            _n_graph = len(_graph_nodes) if isinstance(_graph_nodes, list) else 0
            print(f'    route_used={_actual_route} | chunks={_n_chunks} | graph_nodes={_n_graph}')

            if _n_chunks > 0 and isinstance(_chunks, list):
                _sources = [c.get('metadata', {}).get('source', '?') if isinstance(c, dict) else '?' for c in _chunks[:3]]
                print(f'    top_sources: {_sources}')

            norm = normalize_pipeline_output(raw, mode='v2')

            score = exact_correctness(norm['answer'], item.get('ground_truth_answer', ''))
            recall = retrieval_recall_proxy(
                norm.get('source_ids', []),
                item.get('ground_truth_source_chunks', []),
            )
            halluc = hallucination_flag(norm.get('groundedness_score', 0.0), GROUNDING_THRESHOLD)

            # Verbose scoring
            _ans_preview = norm['answer'][:100] if norm.get('answer') else 'EMPTY'
            print(f'    score={score:.0%} | recall={recall:.0%} | halluc={halluc} | grounded={norm.get("groundedness_score",0):.3f}')
            print(f'    answer: "{_ans_preview}..."')

            entry = {
                'idx': i, 'question': q,
                'answer': norm['answer'],
                'gt': item.get('ground_truth_answer', ''),
                'score_exact': score, 'recall': recall, 'hallucination': halluc,
                'route_requested': route_name,
                'route_actual': norm.get('route', ''),
                'latency_ms': round(elapsed * 1000, 1),
                'groundedness': norm.get('groundedness_score', 0.0),
                'embedding': EMB,
                'category': item.get('category', ''),
                'n_chunks': _n_chunks,
                'n_graph_nodes': _n_graph,
            }
        except Exception as e:
            elapsed = time.perf_counter() - start
            _tb = traceback.format_exc()
            print(f'    ❌ EXCEPTION: {type(e).__name__}: {str(e)[:200]}')
            if 'ConnectionError' in type(e).__name__ or 'timeout' in str(e).lower():
                print(f'    Server may be down. Check /content/server.log')
                print(f'    Last 20 lines of server log:')
                import subprocess
                subprocess.run(['tail', '-20', '/content/server.log'])
            entry = {
                'idx': i, 'question': q, 'error': str(e),
                'route_requested': route_name, 'embedding': EMB,
                'category': item.get('category', ''),
                'score_exact': 0, 'recall': 0, 'hallucination': 1,
                'latency_ms': round(elapsed * 1000, 1), 'groundedness': 0.0,
                'n_chunks': 0, 'n_graph_nodes': 0,
            }

        all_results.append(entry)

        if (i + 1) % 10 == 0:
            ok = [r for r in all_results if r.get('route_requested') == route_name and 'error' not in r]
            err = [r for r in all_results if r.get('route_requested') == route_name and 'error' in r]
            acc = sum(r['score_exact'] for r in ok) / len(ok) if ok else 0
            print(f'\n  ─── Progress [{i+1}/{len(rows)}] acc={acc:.2%} | errors={len(err)} ───\n')

# Save full log
log_path = f'results/colab_eval/query_log_{EMB}_full.jsonl'
with open(log_path, 'w', encoding='utf-8') as f:
    for r in all_results:
        f.write(json.dumps(r, ensure_ascii=False) + '\n')

print(f'\n✅ Done: {len(all_results)} total evaluations → {log_path}')

## 7. Summary Statistics

In [ ]:
import pandas as pd
import numpy as np

df = pd.DataFrame(all_results)
df_ok = df[~df['error'].notna()] if 'error' in df.columns else df
# Filter out errors
df_ok = df[df.apply(lambda r: 'error' not in r or pd.isna(r.get('error')), axis=1)].copy()

# Per-route summary
summary = df.groupby('route_requested').agg(
    count=('score_exact', 'count'),
    accuracy=('score_exact', 'mean'),
    recall=('recall', 'mean'),
    hallucination_rate=('hallucination', 'mean'),
    groundedness=('groundedness', 'mean'),
    latency_p50=('latency_ms', 'median'),
    latency_p95=('latency_ms', lambda x: np.percentile(x, 95)),
).round(4)

print(f'\n=== {EMB.upper()} — Route Comparison ===')
print(summary.to_string())

# Per-category × route
cat_summary = df.groupby(['category', 'route_requested']).agg(
    count=('score_exact', 'count'),
    accuracy=('score_exact', 'mean'),
    recall=('recall', 'mean'),
    hallucination_rate=('hallucination', 'mean'),
).round(4)

print(f'\n=== Per-Category × Route ===')
print(cat_summary.to_string())

# Save
summary.to_csv(f'results/colab_eval/summary_{EMB}.csv')
cat_summary.to_csv(f'results/colab_eval/category_summary_{EMB}.csv')

## 8. Charts for Thesis Report 📊

In [ ]:
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['font.size'] = 12

# ====== Chart 1: Accuracy by Route ======
fig, ax = plt.subplots(figsize=(8, 5))
routes = summary.index.tolist()
acc_vals = summary['accuracy'].values
colors = ['#2196F3', '#4CAF50', '#FF9800']

bars = ax.bar(routes, acc_vals, color=colors[:len(routes)], edgecolor='black', linewidth=0.5)
for bar, val in zip(bars, acc_vals):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f'{val:.1%}', ha='center', va='bottom', fontweight='bold')

ax.set_title(f'Accuracy by Route Strategy — {EMB.upper()}', fontweight='bold', fontsize=14)
ax.set_ylabel('Accuracy')
ax.set_ylim(0, 1.1)
ax.axhline(y=0.8, color='red', linestyle='--', alpha=0.5, label='Target 80%')
ax.legend()
plt.tight_layout()
plt.savefig(f'results/colab_eval/chart_accuracy_{EMB}.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ====== Chart 2: Latency Distribution (Box Plot) ======
fig, ax = plt.subplots(figsize=(8, 5))

route_groups = [df[df['route_requested'] == r]['latency_ms'].dropna().values for r in routes]
bp = ax.boxplot(route_groups, labels=routes, patch_artist=True, showfliers=True)

for patch, color in zip(bp['boxes'], colors[:len(routes)]):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)

ax.set_title(f'Latency Distribution by Route — {EMB.upper()}', fontweight='bold', fontsize=14)
ax.set_ylabel('Latency (ms)')
ax.axhline(y=summary['latency_p95'].max(), color='red', linestyle='--', alpha=0.5, label=f'P95 max: {summary["latency_p95"].max():.0f}ms')
ax.legend()
plt.tight_layout()
plt.savefig(f'results/colab_eval/chart_latency_{EMB}.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ====== Chart 3: Radar Chart — Multi-metric Comparison ======
from math import pi

metrics = ['accuracy', 'recall', 'groundedness']
# Invert hallucination (1 - rate) so higher = better on radar
metric_labels = ['Accuracy', 'Recall', 'Groundedness', '1 - Hallucination']

fig, ax = plt.subplots(figsize=(8, 8), subplot_kw=dict(polar=True))

N = len(metric_labels)
angles = [n / float(N) * 2 * pi for n in range(N)]
angles += angles[:1]

for idx, route in enumerate(routes):
    row = summary.loc[route]
    values = [
        row['accuracy'],
        row['recall'],
        row['groundedness'],
        1 - row['hallucination_rate'],
    ]
    values += values[:1]
    ax.plot(angles, values, 'o-', linewidth=2, label=route, color=colors[idx])
    ax.fill(angles, values, alpha=0.1, color=colors[idx])

ax.set_xticks(angles[:-1])
ax.set_xticklabels(metric_labels, fontsize=11)
ax.set_ylim(0, 1)
ax.set_title(f'Multi-metric Radar — {EMB.upper()}', fontweight='bold', fontsize=14, y=1.08)
ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1))
plt.tight_layout()
plt.savefig(f'results/colab_eval/chart_radar_{EMB}.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ====== Chart 4: Category Breakdown (Grouped Bar) ======
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

categories = df['category'].unique()
x = np.arange(len(categories))
width = 0.25

for metric_idx, (metric, title) in enumerate([('score_exact', 'Accuracy'), ('recall', 'Recall')]):
    ax = axes[metric_idx]
    for r_idx, route in enumerate(routes):
        vals = [df[(df['category'] == cat) & (df['route_requested'] == route)][metric].mean()
                for cat in categories]
        ax.bar(x + r_idx * width, vals, width, label=route, color=colors[r_idx], alpha=0.8)

    ax.set_title(f'{title} by Category — {EMB.upper()}', fontweight='bold')
    ax.set_xticks(x + width)
    ax.set_xticklabels(categories)
    ax.set_ylabel(title)
    ax.set_ylim(0, 1.1)
    ax.legend(fontsize=9)

plt.tight_layout()
plt.savefig(f'results/colab_eval/chart_category_{EMB}.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ====== Chart 5: Heatmap — Category × Route Accuracy ======
import seaborn as sns

pivot = df.pivot_table(values='score_exact', index='category', columns='route_requested', aggfunc='mean')

fig, ax = plt.subplots(figsize=(8, 4))
sns.heatmap(pivot, annot=True, fmt='.2%', cmap='RdYlGn', vmin=0, vmax=1,
            linewidths=1, ax=ax, cbar_kws={'label': 'Accuracy'})
ax.set_title(f'Accuracy Heatmap: Category × Route — {EMB.upper()}', fontweight='bold', fontsize=14)
plt.tight_layout()
plt.savefig(f'results/colab_eval/chart_heatmap_{EMB}.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ====== Chart 6: Error Analysis — Stacked Bar ======
fig, ax = plt.subplots(figsize=(10, 5))

error_types = []
for _, row in df.iterrows():
    if row.get('score_exact', 0) == 0:
        if row.get('hallucination', 0) == 1:
            error_types.append('hallucination')
        elif row.get('recall', 0) == 0:
            error_types.append('retrieval_miss')
        else:
            error_types.append('reasoning_error')
    else:
        error_types.append('correct')

df['error_type'] = error_types

error_pivot = df.groupby(['route_requested', 'error_type']).size().unstack(fill_value=0)
error_colors = {'correct': '#4CAF50', 'hallucination': '#F44336', 'retrieval_miss': '#FF9800', 'reasoning_error': '#9C27B0'}
col_order = [c for c in ['correct', 'retrieval_miss', 'hallucination', 'reasoning_error'] if c in error_pivot.columns]
error_pivot[col_order].plot(kind='bar', stacked=True, ax=ax,
                            color=[error_colors[c] for c in col_order], edgecolor='black', linewidth=0.3)

ax.set_title(f'Error Type Distribution by Route — {EMB.upper()}', fontweight='bold', fontsize=14)
ax.set_ylabel('Count')
ax.set_xlabel('Route')
ax.legend(title='Type', bbox_to_anchor=(1.05, 1))
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig(f'results/colab_eval/chart_errors_{EMB}.png', dpi=150, bbox_inches='tight')
plt.show()

## 9. Top Failed Questions (for Analysis)

In [ ]:
# Show worst questions across all routes
fails = df[df['score_exact'] == 0].copy()
if len(fails) > 0:
    # Count failures per question
    fail_counts = fails.groupby('question').agg(
        fail_routes=('route_requested', lambda x: ', '.join(x)),
        avg_groundedness=('groundedness', 'mean'),
        total_fails=('score_exact', 'count'),
    ).sort_values('total_fails', ascending=False).head(10)

    print(f'=== Top 10 Most Failed Questions ({EMB.upper()}) ===')
    for i, (q, row) in enumerate(fail_counts.iterrows(), 1):
        print(f'\n{i}. [{row["total_fails"]}/3 routes failed] Q: {q[:80]}...')
        print(f'   Failed on: {row["fail_routes"]}')
        print(f'   Avg groundedness: {row["avg_groundedness"]:.3f}')
else:
    print('No failures! All questions answered correctly.')

## 10. Generate Thesis Report

In [ ]:
report_lines = []
report_lines.append(f'# Báo Cáo Đánh Giá RAG Pipeline — {EMB.upper()}')
report_lines.append(f'\n**Ngày:** {time.strftime("%Y-%m-%d %H:%M")}')
report_lines.append(f'**Embedding:** {emb_cfg["model"]} (dim={emb_cfg["dim"]})')
report_lines.append(f'**Số câu hỏi:** {len(rows)}')
report_lines.append(f'**Route strategies:** padded_rag, graph_rag, auto')

report_lines.append('\n## 1. Tổng quan kết quả')
report_lines.append('\n| Route | N | Accuracy | Recall | Hallucination | Groundedness | Latency p50 | Latency p95 |')
report_lines.append('|-------|---|----------|--------|---------------|-------------|-------------|-------------|')
for route in routes:
    route_name = ROUTE_NAMES[route]
    row = summary.loc[route_name]
    report_lines.append(
        f'| {route_name} | {int(row["count"])} '
        f'| {row["accuracy"]:.2%} | {row["recall"]:.2%} '
        f'| {row["hallucination_rate"]:.2%} | {row["groundedness"]:.3f} '
        f'| {row["latency_p50"]:.0f}ms | {row["latency_p95"]:.0f}ms |'
    )

# Best route
best_route = summary['accuracy'].idxmax()
report_lines.append(f'\n**Route tốt nhất theo accuracy:** `{best_route}` ({summary.loc[best_route, "accuracy"]:.2%})')

report_lines.append('\n## 2. Phân tích theo Category')
report_lines.append('\n| Category | Route | N | Accuracy | Recall | Hallucination |')
report_lines.append('|----------|-------|---|----------|--------|---------------|')
for (cat, route), row in cat_summary.iterrows():
    report_lines.append(
        f'| {cat} | {route} | {int(row["count"])} '
        f'| {row["accuracy"]:.2%} | {row["recall"]:.2%} '
        f'| {row["hallucination_rate"]:.2%} |'
    )

report_lines.append('\n## 3. Phân tích lỗi')
if len(fails) > 0:
    error_summary = df['error_type'].value_counts()
    report_lines.append('\n| Loại lỗi | Số lượng | Tỷ lệ |')
    report_lines.append('|----------|----------|-------|')
    for etype, count in error_summary.items():
        report_lines.append(f'| {etype} | {count} | {count/len(df):.2%} |')

report_lines.append('\n## 4. Charts')
report_lines.append(f'\n![Accuracy](chart_accuracy_{EMB}.png)')
report_lines.append(f'![Latency](chart_latency_{EMB}.png)')
report_lines.append(f'![Radar](chart_radar_{EMB}.png)')
report_lines.append(f'![Category](chart_category_{EMB}.png)')
report_lines.append(f'![Heatmap](chart_heatmap_{EMB}.png)')
report_lines.append(f'![Errors](chart_errors_{EMB}.png)')

report_lines.append('\n## 5. Nhận xét & Khuyến nghị')
report_lines.append('\n### Điểm mạnh')
if summary['accuracy'].max() >= 0.8:
    report_lines.append(f'- Route `{best_route}` đạt accuracy ≥ 80% target')
if summary['hallucination_rate'].min() < 0.15:
    report_lines.append(f'- Hallucination rate thấp ({summary["hallucination_rate"].min():.2%})')

report_lines.append('\n### Điểm yếu cần cải thiện')
worst_route = summary['accuracy'].idxmin()
if summary.loc[worst_route, 'accuracy'] < 0.7:
    report_lines.append(f'- Route `{worst_route}` accuracy thấp ({summary.loc[worst_route, "accuracy"]:.2%})')
if summary['latency_p95'].max() > 5000:
    report_lines.append(f'- Latency p95 cao ({summary["latency_p95"].max():.0f}ms) — cần optimize')

report_lines.append('\n### Roadmap cải tiến')
report_lines.append('1. **Quick wins (1-3 ngày):** Tune top_k, adjust groundedness threshold')
report_lines.append('2. **Medium (1-2 tuần):** Thêm reranker, query rewriting')
report_lines.append('3. **Long-term (2-6 tuần):** Hard negative mining, graph reasoning optimization')

# Write report
report_text = '\n'.join(report_lines)
with open(f'results/colab_eval/thesis_report_{EMB}.md', 'w', encoding='utf-8') as f:
    f.write(report_text)

print(report_text)

## 11. Download Results

In [ ]:
from google.colab import files
import glob

print(f'=== Files to download ({EMB.upper()}) ===')
all_files = glob.glob('results/colab_eval/*')
for f in sorted(all_files):
    print(f'  {f}')

# Download all
for f in all_files:
    try:
        files.download(f)
    except Exception as e:
        print(f'  Skip {f}: {e}')

## 12. Free LLM APIs cho HyDE (Hypothetical Document Embedding)

Danh sách các nguồn free LLM API có thể dùng cho HyDE generation, không cần thẻ tín dụng:

| # | Provider | Free Model | Rate Limit | Link |
|---|----------|-----------|------------|------|
| 1 | **Groq** | llama-3.3-70b-versatile | 30 req/min | https://console.groq.com |
| 2 | **Google AI Studio** | gemini-2.0-flash | 15 req/min | https://aistudio.google.com |
| 3 | **OpenRouter** | nhiều model free | 20 req/min | https://openrouter.ai |
| 4 | **Cerebras** | llama-3.3-70b | 30 req/min | https://cloud.cerebras.ai |
| 5 | **Together AI** | nhiều model free | 60 req/min | https://api.together.xyz |
| 6 | **GitHub Models** | gpt-4o-mini, phi-3 | varies | https://github.com/marketplace/models |
| 7 | **Cloudflare Workers AI** | nhiều model | 10K neurons/day | https://ai.cloudflare.com |
| 8 | **HuggingFace Serverless** | nhiều model | 1000 req/day | https://huggingface.co/docs/api-inference |

**Tất cả đều OpenAI-compatible endpoint** — chỉ cần đổi `base_url` + `api_key`.

**Khuyến nghị cho HyDE:**
- **Groq** (nhanh nhất, latency ~200ms) — đã tích hợp sẵn trong codebase
- **Google AI Studio** (miễn phí, chất lượng tốt) — cần `GOOGLE_API_KEY`
- **OpenRouter** (nhiều lựa chọn model free) — cần `OPENROUTER_API_KEY`